# Lab 1 – Text Dataset Loading, Tokenization & DataLoader

**Modifications from original:**
- Dataset: `ag_news` instead of WikiText-2
- Tokenizer: `distilbert-base-uncased` instead of GPT-2
- Block size: 256 tokens instead of 128
- Added: vocab size info, decode sanity check, attention mask stats

In [ ]:
!pip install datasets

In [ ]:
!pip install transformers datasets torch

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader
import torch

In [ ]:
# 1. Load AG News dataset (news topic classification — 4 categories, ~120k training articles)
# We only use the 'text' field for language-model-style tokenization
dataset = load_dataset("ag_news", split="train")
print(f"Number of examples in dataset: {len(dataset)}")
print(f"Features: {dataset.features}")
print(f"\nSample text:\n{dataset[0]['text']}")

In [ ]:
# 2. Initialize DistilBERT tokenizer (bidirectional, subword BPE, different from GPT-2's causal BPE)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Special tokens: PAD={tokenizer.pad_token!r}, CLS={tokenizer.cls_token!r}, SEP={tokenizer.sep_token!r}")

In [ ]:
# 3. Tokenize using batched .map() — strip labels, keep only text
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=False,
        return_special_tokens_mask=False,
        return_token_type_ids=False  # ← this stops DistilBERT from generating it at all
    )

tokenized_ds = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text", "label"]
)

first_ids = tokenized_ds[0]["input_ids"]
print(f"Columns in dataset: {tokenized_ds.column_names}")  # should only show: input_ids, attention_mask
print(f"First example token count: {len(first_ids)}")
print(f"First 20 token IDs: {first_ids[:20]}")
print(f"Decoded back: {tokenizer.decode(first_ids[:20])}")

In [ ]:
# 4. Group into fixed-length blocks of 256 tokens
block_size = 256

def group_texts(examples):
    concatenated_inputs = sum(examples["input_ids"], [])
    concatenated_masks  = sum(examples["attention_mask"], [])

    total_len = (len(concatenated_inputs) // block_size) * block_size
    concatenated_inputs = concatenated_inputs[:total_len]
    concatenated_masks  = concatenated_masks[:total_len]

    result_input_ids = [concatenated_inputs[i:i+block_size] for i in range(0, total_len, block_size)]
    result_masks     = [concatenated_masks[i:i+block_size]  for i in range(0, total_len, block_size)]

    return {"input_ids": result_input_ids, "attention_mask": result_masks}

lm_ds = tokenized_ds.map(group_texts, batched=True, batch_size=1000)
print(f"Total LM training sequences (block_size={block_size}): {len(lm_ds)}")

In [ ]:
# 5. Build a DataLoader with a collate function
def collate_fn(batch):
    input_ids = torch.tensor([ex["input_ids"]      for ex in batch], dtype=torch.long)
    attn_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
    # For masked/causal LM, labels == input_ids; model handles loss masking internally
    return {"input_ids": input_ids, "attention_mask": attn_mask, "labels": input_ids.clone()}

train_loader = DataLoader(lm_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
print(f"Number of batches: {len(train_loader)}")

In [ ]:
# 6. Verify batch shapes and inspect attention mask statistics
for batch in train_loader:
    ids   = batch["input_ids"]
    masks = batch["attention_mask"]
    labels = batch["labels"]

    print(f"input_ids shape : {ids.shape}")
    print(f"attention_mask  : {masks.shape}")
    print(f"labels shape    : {labels.shape}")

    # Extra: fraction of real tokens vs padding in this batch
    real_tokens = masks.sum().item()
    total_tokens = masks.numel()
    print(f"Real-token ratio: {real_tokens}/{total_tokens} = {real_tokens/total_tokens:.2%}")

    # Decode the first sequence of the batch back to readable text
    print(f"\nDecoded sequence 0 (first 50 tokens):\n{tokenizer.decode(ids[0][:50])}")
    break

print("\nDataLoader is working correctly!")